# Airbnb Thessaloniki: Data Cleaning

**Source:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), Thessaloniki, Central Macedonia, Greece, data release of **24 September 2026**.

**Files:**
- `listings.csv`: one row per listing (detailed version, 90 columns)
- `calendar.csv`: availability (`t`/`f`) and minimum/maximum nights per listing per day; there is no price column
- `reviews.csv`: one row per guest review

The column definitions are in the Inside Airbnb data dictionary (linked on the *Get the Data* page).

## Introduction

*TODO (write at the end, Section 9): the goal of this notebook in one or two sentences and the main problems found in the raw data.*

### How this notebook is organised

*TODO (fill in at the end):*

1. Loading the data
2. First look: what is one row in each table?
3. Shape, dtypes and missing values
   - 3b. Empty and constant columns
4. Dtype conversion
5. Missing values by group
6. Keys, duplicates and referential integrity
7. Cross-table consistency
8. Value ranges, typos and date order
9. Export and summary

Imports, and show all columns: `listings` has 90 and pandas would hide most of them.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Loading the data

Load the three raw CSV files. These DataFrames are never changed; every fix will be done on a `_clean` copy.

In [2]:
listings = pd.read_csv("data/listings.csv")
calendar = pd.read_csv("data/calendar.csv")
reviews = pd.read_csv("data/reviews.csv")

Put the three tables in a dict, so every check can run in one loop instead of three identical cells.

In [3]:
dataframes = {
    "listings": listings,
    "calendar": calendar,
    "reviews": reviews
}

## 2. First look: what is one row in each table?

**Hypothesis:** `listings` has one row per listing, `calendar` has one row per listing per day for the next year (so about 365 rows per listing), and `reviews` has one row per guest review.

Size of each table (rows, columns).

In [4]:
for name, df in dataframes.items():
    print(f"{name} shape: {df.shape}")

listings shape: (4961, 90)
calendar shape: (1810765, 5)
reviews shape: (244369, 6)


Calendar rows per listing: tests the hypothesis that each listing has one row per day.

In [5]:
calendar.shape[0] / listings.shape[0]

365.0

**Findings:**
- `listings` has 4,961 rows and 90 columns, `calendar` has 1,810,765 rows and `reviews` has 244,369 rows.
- `calendar` rows / `listings` rows = 365.0. This means every listing has exactly one year of daily rows, which supports the hypothesis.
- But `calendar.date` has 366 distinct dates, so not all listings start on the same day. This is checked in Section 6.

### Grain of each table

| table | one row is | candidate key | status |
|---|---|---|---|
| `listings` | one Airbnb listing | `id` | assumed |
| `calendar` | one listing on one date | (`listing_id`, `date`) | assumed |
| `reviews` | one guest review | `id` | assumed |

**Relationships:** `calendar.listing_id` and `reviews.listing_id` both point to `listings.id`. The status will be checked in Section 6.

## 3. Shape, dtypes and missing values

`info()` gives a quick overview of each table: number of rows, the dtype of every column and how many values are not missing.

In [6]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 4961 entries, 0 to 4960
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            4961 non-null   int64  
 1   listing_url                                   4961 non-null   str    
 2   scrape_id                                     4961 non-null   int64  
 3   last_scraped                                  4961 non-null   str    
 4   source                                        435 non-null    str    
 5   name                                          4961 non-null   str    
 6   description                                   4960 non-null   str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   4961 non-null   str    
 9   host_id                                       4961 non-null   int64  
 10 

In [7]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 1810765 entries, 0 to 1810764
Data columns (total 5 columns):
 #   Column          Dtype
---  ------          -----
 0   listing_id      int64
 1   date            str  
 2   available       str  
 3   minimum_nights  int64
 4   maximum_nights  int64
dtypes: int64(3), str(2)
memory usage: 69.1 MB


In [8]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 244369 entries, 0 to 244368
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   listing_id     244369 non-null  int64
 1   id             244369 non-null  int64
 2   date           244369 non-null  str  
 3   reviewer_id    244369 non-null  int64
 4   reviewer_name  244369 non-null  str  
 5   comments       244355 non-null  str  
dtypes: int64(3), str(3)
memory usage: 11.2 MB


`info()` is hard to read with 90 columns, so `column_summary` builds one row per column with its dtype, the number and percentage of NaN, and the number of distinct values. The most empty columns come first.

In [9]:
def column_summary(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "n_nan": df.isna().sum(),
        "pct_nan": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique()
    })
    return summary.sort_values("n_nan", ascending=False)

for name, df in dataframes.items():
    print(name)
    display(column_summary(df))

listings


,dtype,n_nan,pct_nan,n_unique
neighborhood_overview,float64,4961,100.0,0
host_since,float64,4961,100.0,0
host_response_time,float64,4961,100.0,0
host_thumbnail_url,float64,4961,100.0,0
host_acceptance_rate,float64,4961,100.0,0
...,...,...,...,...
availability_30,int64,0,0.0,31
calculated_host_listings_count,int64,0,0.0,35
calculated_host_listings_count_entire_homes,int64,0,0.0,37
calculated_host_listings_count_private_rooms,int64,0,0.0,9


calendar


,dtype,n_nan,pct_nan,n_unique
listing_id,int64,0,0.0,4961
date,str,0,0.0,366
available,str,0,0.0,2
minimum_nights,int64,0,0.0,36
maximum_nights,int64,0,0.0,131


reviews


,dtype,n_nan,pct_nan,n_unique
comments,str,14,0.0,227596
listing_id,int64,0,0.0,4390
id,int64,0,0.0,244369
date,str,0,0.0,4525
reviewer_id,int64,0,0.0,194620
reviewer_name,str,0,0.0,34248


Keep only the `listings` columns that have at least one missing value.

In [10]:
s = column_summary(listings)
s[s["n_nan"] > 0]

,dtype,n_nan,pct_nan,n_unique
neighborhood_overview,float64,4961,100.0,0
host_since,float64,4961,100.0,0
host_response_time,float64,4961,100.0,0
host_thumbnail_url,float64,4961,100.0,0
host_acceptance_rate,float64,4961,100.0,0
host_response_rate,float64,4961,100.0,0
host_verifications,float64,4961,100.0,0
neighbourhood,float64,4961,100.0,0
host_total_listings_count,float64,4961,100.0,0
host_neighbourhood,float64,4961,100.0,0


The `str` columns of `listings` with their first three values, to see which ones are really text and which ones hide dates, flags, numbers or lists.

In [11]:
obj_cols = s[s["dtype"] == "str"].index
listings[obj_cols].head(3).T

,0,1,2
source,NaN,NaN,NaN
host_about,NaN,Με όραμα την Ελληνική Φιλοξενία και μοναδικό σ...,NaN
host_location,NaN,"Thessaloniki, Greece",NaN
last_review,2026-09-06,2026-08-20,NaN
first_review,2026-03-15,2026-03-08,NaN
price,$96.60,$40.00,$60.00
price_quote_raw,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...","{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...","{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""..."
price_quote_checkin_date,2026-09-24,2026-09-30,2026-10-03
price_quote_checkout_date,2026-09-26,2026-10-01,2026-10-05
license,00003427019,3672670,00003673975


In [12]:
# currency named inside each price quote (JSON text)
currency = listings["price_quote_raw"].str.extract(r'"currency":\s*"(\w+)"')[0]
currency.value_counts(dropna=False)

0
EUR    4632
NaN     329
Name: count, dtype: int64

**Findings:** the `str` columns fall into five groups.
- **Dates stored as text** (`YYYY-MM-DD`): `last_scraped`, `calendar_last_scraped`, `first_review`, `last_review`, `price_quote_checkin_date`, `price_quote_checkout_date`. Converted in Section 4a.
- **Yes/no flags stored as `t`/`f`**: `host_is_superhost`, `host_identity_verified`, `host_has_profile_pic`, `has_availability`.
- **A number hidden in text**: `price` (`$96.60`) and `bathrooms_text` (`1 bath`, `1.5 baths`, `Half-bath`).
- **A list or JSON stored as text**: `amenities` (a list of amenities) and `price_quote_raw` (the full price quote as JSON).
- **Real text** that stays as it is: names, descriptions, URLs and categories (`name`, `description`, `host_name`, `host_about`, `host_location`, `room_type`, `property_type`, `neighbourhood_cleansed`, the `*_url` columns, `source`).

Two things to keep in mind:
- `price` is written with a `$` sign, but `price_quote_raw` says `"currency": "EUR"`. Every quote that names a currency says EUR (checked below), so the `$` is only a display symbol and the prices are in **euros**.
- `license` is a registration number, so it stays text. Its values are not written the same way (`00003427019` with leading zeros, `3672670` without). This is looked at in Section 5.

`float64` columns with no values at all. pandas gives `float64` to a completely empty column, so here the dtype says nothing about the content. These columns are handled in Section 3b.

In [13]:
s[(s["dtype"] == "float64") & (s["pct_nan"] == 100)]

,dtype,n_nan,pct_nan,n_unique
neighborhood_overview,float64,4961,100.0,0
host_since,float64,4961,100.0,0
host_response_time,float64,4961,100.0,0
host_thumbnail_url,float64,4961,100.0,0
host_acceptance_rate,float64,4961,100.0,0
host_response_rate,float64,4961,100.0,0
host_verifications,float64,4961,100.0,0
neighbourhood,float64,4961,100.0,0
host_total_listings_count,float64,4961,100.0,0
host_neighbourhood,float64,4961,100.0,0


Columns with only one distinct value. `nunique()` ignores NaN, so a column with one value plus NaN also shows up here. Whether each one is truly constant is checked in Section 3b.

In [14]:
s[s["n_unique"] == 1]

,dtype,n_nan,pct_nan,n_unique
source,str,4526,91.2,1
has_availability,str,11,0.2,1
scrape_id,int64,0,0.0,1


The id columns: how many digits they have, and whether they would survive a conversion to `float64`.

In [15]:
for name, df in dataframes.items():
    for col in df.columns:
        if col.endswith("id"):
            biggest = df[col].max()
            print(name, col, df[col].dtype, len(str(biggest)), "digits,",
                  "safe as float:", int(float(biggest)) == biggest , "\n")

listings id int64 19 digits, safe as float: False 

listings scrape_id int64 14 digits, safe as float: True 

listings host_id int64 19 digits, safe as float: False 

listings host_profile_id int64 19 digits, safe as float: False 

calendar listing_id int64 19 digits, safe as float: False 

reviews listing_id int64 19 digits, safe as float: False 

reviews id int64 19 digits, safe as float: False 

reviews reviewer_id int64 19 digits, safe as float: False 



In [16]:
# how many ids are above 2**53, and how many change after a round trip through float
for name, df in dataframes.items():
    for col in df.columns:
        if col.endswith("id"):
            digits = df[col].astype(str).str.len()
            n_big = (df[col] > 2**53).sum()
            n_changed = (df[col].astype(float).astype("int64") != df[col]).sum()
            print(f"{name}.{col}: {digits.min()}-{digits.max()} digits, "
                  f"{n_big:,} values > 2**53, {n_changed:,} change through float")

listings.id: 6-19 digits, 3,812 values > 2**53, 3,796 change through float
listings.scrape_id: 14-14 digits, 0 values > 2**53, 0 change through float
listings.host_id: 6-19 digits, 119 values > 2**53, 119 change through float
listings.host_profile_id: 19-19 digits, 4,961 values > 2**53, 4,952 change through float
calendar.listing_id: 6-19 digits, 1,391,380 values > 2**53, 1,385,540 change through float
reviews.listing_id: 6-19 digits, 100,278 values > 2**53, 99,696 change through float
reviews.id: 7-19 digits, 194,652 values > 2**53, 193,421 change through float
reviews.reviewer_id: 4-19 digits, 2,266 values > 2**53, 2,263 change through float


**Findings:** the `id` in `listings` has between 6 and 19 digits. 3,812 of the 4,961 ids are larger than 2⁵³, the largest integer float64 stores exactly, and a round trip through float changes 3,796 of them. The same problem appears in `host_id`, `host_profile_id`, `calendar.listing_id`, `reviews.listing_id`, `reviews.id` and `reviews.reviewer_id`; only `scrape_id` (14 digits) is safe.

int64 holds them correctly, so the columns stay int64. But any step that introduces a NaN into an id column (e.g. a merge) would silently turn it into float and corrupt these ids, so I check the id dtypes after every merge.

### Results

- **`listings`**:
  - Missing values: 14 columns are 100% empty. `source` is 91% NaN, `host_about` 55% and `host_location` 37%. The review columns (`first_review`, `last_review`, `reviews_per_month`, `review_scores_*`) are all missing in 571 rows (11.5%), and the price columns (`price`, `price_quote_*`, `estimated_revenue_l365d`) in 300–334 rows (about 6.7%). Smaller gaps: `license` (105), `beds` (48), `has_availability` (11), the `*_nights` columns (3), `bathrooms_text` and `description` (1 each).
  - Wrong dtypes: the dates (`last_scraped`, `calendar_last_scraped`, `first_review`, `last_review`, `price_quote_checkin_date`, `price_quote_checkout_date`) are text. The t/f columns (`host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `has_availability`) are text. `price` is text with a `$` sign. `bathrooms_text` and `amenities` hide a number and a list inside text. `minimum_nights` and `maximum_nights` are float only because of 3 NaN.
- **`calendar`**: no missing values. `date` and `available` (t/f) are text.
- **`reviews`**: 14 reviews have no `comments`. `date` is text.
- **All tables**: the id columns are int64 with up to 19 digits and would be corrupted if they became float.

Nothing is fixed yet. This list is the work for Sections 3b–5.

Use the copy method to create a copy of the original dataframes for cleaning and processing. This ensures that the original data remains unchanged and can be referenced later if needed.

In [17]:
listings_clean = listings.copy()
calendar_clean = calendar.copy()
reviews_clean = reviews.copy()

dataframes_clean = {
    "listings": listings_clean, "calendar": calendar_clean, "reviews": reviews_clean
}



### 3b. Empty and constant columns

Find the `listings` columns that are 100% empty (NaN in every row).

In [18]:
empty_cols = listings_clean.columns[listings_clean.isna().all()]
print(len(empty_cols))
list(empty_cols)

14


['neighborhood_overview',
 'host_since',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_thumbnail_url',
 'host_neighbourhood',
 'host_total_listings_count',
 'host_verifications',
 'neighbourhood',
 'neighbourhood_group_cleansed',
 'bathrooms',
 'calendar_updated',
 'instant_bookable']

Before dropping an empty column, check whether the same information exists in another column:

| empty column | related column | has values? | decision |
|---|---|---|---|
| `bathrooms` | `bathrooms_text` | yes (1 NaN) | drop; the number can be extracted from `bathrooms_text` in Section 4 |
| `host_since` | `hosts_time_as_host_years` / `_months` | yes (0 NaN) | drop; the time as host is still known in years and months |
| `neighbourhood` | `neighbourhood_cleansed` | yes (0 NaN, 7 areas) | drop; `neighbourhood_cleansed` is used instead |


In [19]:
listings_clean[["bathrooms", "bathrooms_text"]].head(10)


,bathrooms,bathrooms_text
0,NaN,1 bath
1,NaN,1 bath
2,NaN,1 bath
3,NaN,1 bath
4,NaN,1 bath
5,NaN,1 bath
6,NaN,1 bath
7,NaN,1 bath
8,NaN,1 bath
9,NaN,1 bath


In [20]:
listings_clean[["bathrooms", "bathrooms_text"]].isna().sum()


bathrooms         4961
bathrooms_text       1
dtype: int64

In [21]:
listings_clean["bathrooms_text"].value_counts().head(10)


bathrooms_text
1 bath            4325
1.5 baths          251
2 baths            199
1 private bath     114
1 shared bath       17
3 baths             11
0 baths             11
2.5 baths           11
Half-bath            8
4 baths              4
Name: count, dtype: int64

In [22]:
missing_bathrooms = listings_clean["bathrooms"].isna()

In [23]:
print(listings_clean[["neighbourhood", "neighbourhood_cleansed"]].isna().sum())
listings_clean["neighbourhood_cleansed"].value_counts()


neighbourhood             4961
neighbourhood_cleansed       0
dtype: int64


neighbourhood_cleansed
Thessaloniki              4404
Kalamaria                  177
Neapolis – Sykeon          125
Pilea – Chortiatis          74
Pavlou Mela                 68
Ampelokipon – Menemeni      65
Kordelio – Evosmos          48
Name: count, dtype: int64

**Check:** `host_since` is empty in all rows, but `hosts_time_as_host_years` and `_months` have no NaN, so the time as host is still known. `neighbourhood` is empty in all rows, while `neighbourhood_cleansed` has no NaN and 7 areas. Thessaloniki holds 4,404 of the 4,961 listings (89%), and the other 6 areas have between 48 and 177 each. Both empty columns can be dropped without losing information.


In [24]:
host_cols = ["host_since", "hosts_time_as_host_years", "hosts_time_as_host_months"]
print(listings_clean[host_cols].isna().sum())
listings_clean[host_cols].head()

host_since                   4961
hosts_time_as_host_years        0
hosts_time_as_host_months       0
dtype: int64


,host_since,hosts_time_as_host_years,hosts_time_as_host_months
0,NaN,0,11
1,NaN,7,1
2,NaN,0,4
3,NaN,9,4
4,NaN,9,4


The single-value columns found in Section 3 are checked again with `nunique(dropna=False)`, because `nunique()` ignores NaN.
- Truly constant (to drop): `scrape_id`, the same value in all 4,961 rows.
- One value + NaN (kept for now, acts as a yes/no flag): `source` ("previous scrape" in 435 rows, NaN in 4,526) and `has_availability` ("t" in 4,950 rows, NaN in 11).


In [25]:
single_cols = s[s["n_unique"] == 1].index
listings_clean[single_cols].nunique(dropna=False)


source              2
has_availability    2
scrape_id           1
dtype: int64

In [26]:
for col in ["source", "has_availability"]:
    print(listings_clean[col].value_counts(dropna=False), "\n")


source
NaN                4526
previous scrape     435
Name: count, dtype: int64 

has_availability
t      4950
NaN      11
Name: count, dtype: int64 



In [27]:
listings_clean.shape

(4961, 90)

In [28]:
cols_to_drop = list(empty_cols) + ["scrape_id"]
listings_clean = listings_clean.drop(columns=cols_to_drop)
dataframes_clean["listings"] = listings_clean
print(len(cols_to_drop), "columns dropped")
listings_clean.shape


15 columns dropped


(4961, 75)

**Dropped columns (15):**
- 14 columns that are 100% empty: `neighborhood_overview`, `host_since`, `host_response_time`, `host_response_rate`, `host_acceptance_rate`, `host_thumbnail_url`, `host_neighbourhood`, `host_total_listings_count`, `host_verifications`, `neighbourhood`, `neighbourhood_group_cleansed`, `bathrooms`, `calendar_updated`, `instant_bookable`. None of them holds information that is not in another column: the number of bathrooms is in `bathrooms_text`, the time as host is in `hosts_time_as_host_years` / `_months`, and the area is in `neighbourhood_cleansed`.
- `scrape_id`: the same value in all 4,961 rows.

**Kept:** `source` and `has_availability` have one value plus NaN, so the NaN separates the listings into two groups. Their meaning is checked in Section 5.

`listings_clean` now has 4,961 rows and 75 columns. No rows were dropped.


## 4. Dtype conversion

### 4a. Dates

**Hypothesis:** all date columns are text in the format `YYYY-MM-DD`, so `pd.to_datetime` converts them without creating new NaN. The NaN count before and after the conversion should be the same.


The same check for the `date` column of `calendar` and `reviews`.
